# DATA Preparation


In [148]:
#imports
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## The molecular Dataset

In [149]:
df = pd.read_csv("X_train/clinical_train.csv")
df_eval = pd.read_csv("X_test/clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv("X_train/molecular_train.csv")
maf_eval = pd.read_csv("X_test/molecular_test.csv")

target_df = pd.read_csv("./target_train.csv")

### Some simple processing

In [150]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

#we swap os_status and os_years
target_df['OS_STATUS'], target_df['OS_YEARS'] = target_df['OS_YEARS'], target_df['OS_STATUS']

In [151]:
def extract_cytogenetic_features(df):
    """Extract features from cytogenetics column"""
    
    # Initialize new columns
    df['is_normal'] = df['CYTOGENETICS'].fillna('').astype(str).str.startswith('46').astype(int)
    df['has_deletion'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('del').astype(int)
    df['has_translocation'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('t\(').astype(int)
    df['has_inversion'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('inv').astype(int)
    df['has_addition'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('add').astype(int)
    
    # Check for specific chromosome abnormalities
    df['has_chr7_abnormal'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('-7|del\(7\)').astype(int)
    df['has_chr5_abnormal'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('-5|del\(5\)').astype(int)
    df['has_trisomy8'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('\+8').astype(int)
    
    # Count total abnormalities
    df['total_abnormalities'] = (
        df['has_deletion'] + 
        df['has_translocation'] + 
        df['has_inversion'] + 
        df['has_addition'] +
        df['has_chr7_abnormal'] +
        df['has_chr5_abnormal'] +
        df['has_trisomy8']
    )
    
    return df

# Apply to both training and test sets
df = extract_cytogenetic_features(df)
df_eval = extract_cytogenetic_features(df_eval)

# Let's see the distribution of the new features
for col in ['is_normal', 'has_deletion', 'has_translocation', 'has_inversion', 
            'has_addition', 'has_chr7_abnormal', 'has_chr5_abnormal', 
            'has_trisomy8', 'total_abnormalities']:
    print(f"\nFeature: {col}")
    print(df[col].value_counts())


Feature: is_normal
is_normal
1    2303
0    1020
Name: count, dtype: int64

Feature: has_deletion
has_deletion
0    2709
1     614
Name: count, dtype: int64

Feature: has_translocation
has_translocation
0    3133
1     190
Name: count, dtype: int64

Feature: has_inversion
has_inversion
0    3279
1      44
Name: count, dtype: int64

Feature: has_addition
has_addition
0    3171
1     152
Name: count, dtype: int64

Feature: has_chr7_abnormal
has_chr7_abnormal
0    3104
1     219
Name: count, dtype: int64

Feature: has_chr5_abnormal
has_chr5_abnormal
0    2940
1     383
Name: count, dtype: int64

Feature: has_trisomy8
has_trisomy8
0    3093
1     230
Name: count, dtype: int64

Feature: total_abnormalities
total_abnormalities
0    2344
1     463
2     295
3     127
4      74
5      18
6       2
Name: count, dtype: int64


<>:7: SyntaxWarning: invalid escape sequence '\('
<>:12: SyntaxWarning: invalid escape sequence '\('
<>:13: SyntaxWarning: invalid escape sequence '\('
<>:14: SyntaxWarning: invalid escape sequence '\+'
<>:7: SyntaxWarning: invalid escape sequence '\('
<>:12: SyntaxWarning: invalid escape sequence '\('
<>:13: SyntaxWarning: invalid escape sequence '\('
<>:14: SyntaxWarning: invalid escape sequence '\+'
C:\Users\marec\AppData\Local\Temp\ipykernel_27748\1244646303.py:7: SyntaxWarning: invalid escape sequence '\('
  df['has_translocation'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('t\(').astype(int)
C:\Users\marec\AppData\Local\Temp\ipykernel_27748\1244646303.py:12: SyntaxWarning: invalid escape sequence '\('
  df['has_chr7_abnormal'] = df['CYTOGENETICS'].fillna('').astype(str).str.contains('-7|del\(7\)').astype(int)
C:\Users\marec\AppData\Local\Temp\ipykernel_27748\1244646303.py:13: SyntaxWarning: invalid escape sequence '\('
  df['has_chr5_abnormal'] = df['CYTOGENETICS'].

In [152]:
def extract_cell_proportions(cytogenetics_str):
    """Extract the proportion of abnormal cells from cytogenetics string"""
    if pd.isna(cytogenetics_str):
        return 0, 0
    
    # Split different cell populations
    populations = str(cytogenetics_str).split('/')
    total_cells = 0
    abnormal_cells = 0
    
    for pop in populations:
        # Extract number of cells in square brackets
        cells = re.findall(r'\[(\d+)\]', pop)
        if cells:
            count = int(cells[0])
            total_cells += count
            # If not starting with 46, it's abnormal
            if not pop.strip().startswith('46'):
                abnormal_cells += count
    
    if total_cells == 0:
        return 0, 0
        
    return abnormal_cells, total_cells

def add_severity_features(df):
    """Add severity features based on cell proportions"""
    import re
    
    # Extract cell counts
    cell_counts = df['CYTOGENETICS'].apply(extract_cell_proportions)
    
    # Calculate proportion of abnormal cells
    df['abnormal_cell_count'] = cell_counts.apply(lambda x: x[0])
    df['total_cell_count'] = cell_counts.apply(lambda x: x[1])
    df['abnormal_cell_proportion'] = df.apply(
        lambda row: row['abnormal_cell_count'] / row['total_cell_count'] 
        if row['total_cell_count'] > 0 else 0, 
        axis=1
    )
    
    return df

# Apply to both training and test sets
df = add_severity_features(df)
df_eval = add_severity_features(df_eval)

In [153]:
df.drop(columns=['CYTOGENETICS'], inplace=True)
df_eval.drop(columns=['CYTOGENETICS'], inplace=True)

In [154]:
# Align df and target_df by ID
df.dropna(inplace=True)
target_df.dropna(inplace=True)

# Only keep IDs present in both df and target_df
common_ids = df['ID'].isin(target_df['ID'])
df_aligned = df[common_ids].copy()

# Align target_df to only those IDs present in df_aligned
target_common_ids = target_df['ID'].isin(df_aligned['ID'])
target_df_aligned = target_df[target_common_ids].copy()

# Now align both dataframes by ID
df_aligned = df_aligned.set_index('ID').loc[target_df_aligned['ID']].reset_index()
df_aligned = df_aligned.drop(columns=['ID', 'CENTER'])

target_df_aligned.drop(columns=['ID'], inplace=True)

if 'CENTER' in df_eval.columns:
	df_eval_aligned = df_eval.drop(columns=['CENTER'])
else:
	df_eval_aligned = df_eval.copy()

# Now split
X_train, X_test, y_train, y_test = train_test_split(df_aligned, target_df_aligned, test_size=0.3, random_state=42)

In [155]:
y_train_struct = Surv.from_dataframe('OS_YEARS', 'OS_STATUS', y_train)
y_test_struct = Surv.from_dataframe('OS_YEARS', 'OS_STATUS', y_test)

# Machine Learning

In [156]:
#Importing the necessary libraries

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis , IPCRidge
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv


In [ ]:
from sksurv.meta import EnsembleSelectionRegressor


cox = CoxPHSurvivalAnalysis()
PH = CoxnetSurvivalAnalysis()
ridge = IPCRidge()

base_estimators = [("cox", cox),("PH", PH), ]#("ridge", ridge)]

ensemble = EnsembleSelectionRegressor(
    base_estimators=base_estimators,
    # Pass structured array to the scorer
    scorer=lambda est, X, y: concordance_index_ipcw(y, y, est.predict(X), tau=7)[0],
    n_estimators=1,  # fraction of estimators to retain
    n_jobs=800,         # use all CPUs
    min_score=0.75
)
ensemble.fit(X_train, y_train_struct)

ensemble_cindex_train = concordance_index_ipcw(y_train_struct, y_train_struct, ensemble.predict(X_train), tau=7)[0]
ensemble_cindex_test = concordance_index_ipcw(y_test_struct, y_test_struct, ensemble.predict(X_test), tau=7)[0]
print(f"EnsembleSelectionRegressor Concordance Index IPCW on train: {ensemble_cindex_train:.4f}")
print(f"EnsembleSelectionRegressor Concordance Index IPCW on test: {ensemble_cindex_test:.4f}")

EnsembleSelectionRegressor Concordance Index IPCW on train: 0.6958
EnsembleSelectionRegressor Concordance Index IPCW on test: 0.6739
